In [1]:
import numpy as np
import pdfplumber
import pandas as pd
import re

In [2]:
def cargar_pdf(ruta_pdf: str):
    f = open(ruta_pdf, "rb")
    pdf = pdfplumber.open(f)
    return pdf, f

def extraer_texto_paginas(pdf) -> str:
    texto = ""
    for p in pdf.pages:
        contenido = p.extract_text()
        if contenido:
            texto += contenido + "\n"
    return texto

def limpiar_lineas(texto: str) -> list:
    lineas = [line.strip() for line in texto.split("\n") if line.strip()]
    return lineas

def extraer_transacciones(lineas: list) -> pd.DataFrame:
    patron = re.compile(
        r"(\d{2}/\d{2})\s+(\d+)\s+(.*?)\s+(\d{1,3}(?:\.\d{3})*,\d{2})"
    )

    registros = []
    for linea in lineas:
        match = patron.search(linea)
        if match:
            fecha, vale, establecimiento, valor = match.groups()
            registros.append({
                "fecha": fecha,
                "vale": vale,
                "establecimiento": establecimiento,
                "valor": float(
                    valor.replace(".", "").replace(",", ".")
                )
            })
    return pd.DataFrame(registros)

def cargar_estado_cuenta(ruta_pdf: str) -> pd.DataFrame:
    pdf, f = cargar_pdf(ruta_pdf)
    texto = extraer_texto_paginas(pdf)
    lineas = limpiar_lineas(texto)
    df = extraer_transacciones(lineas)
    pdf.close()
    f.close()
    return df

if __name__ == "__main__":
    ruta = "d:/proyectos_py/presupuesto/data/titanium.pdf"
    df = cargar_estado_cuenta(ruta)
    print(df)



    fecha     vale                       establecimiento    valor
0   30/05  8119543   DIFERIDO INTERNACIONAL ONLINE (1/6)   671.71
1   30/05  8119546   DIFERIDO INTERNACIONAL ONLINE (1/6)   327.11
2   28/11   994746   PTP - UNIVERSIDAD INTERNACIO (7/24)   110.80
3   06/01    56040  PTP - SWEADEN COMPANIA DE SE (FINAL)   100.00
4   14/05        7        DTF BLUECARD ECUADOR S A (2/3)    72.66
5   20/05   224650                 YAG*NUESTRO LEGADO. -    27.00
6   23/05     3333                                KATARI    50.00
7   22/05  2031209                          UBER *TRIP H     3.28
8   22/05  2031209             RET IVA SERV DIGITAL 100%     0.49
9   19/05  1680271                        ROYALTEX S.A U   173.97
10  12/06  4685445                       DTV*DIRECTVGO 4    19.99
11  12/06  4685445             RET IVA SERV DIGITAL 100%     3.00
12  08/06  4067995                 Google Clash Royale 6     8.49
13  05/06  3699772                          UBER RIDES S     4.65
14  05/06 

In [4]:
df.to_excel("estado_cuenta.xlsx", index=False)